# BLIP-base: stock captioning, then fine-tune on devotional images

Two parts:
1. **Part 1 (free, no training)**: test BLIP-base's stock captions, unconditioned and
   prompt-primed, on temple/devotional images. Check whether this alone is good enough
   before spending any time training.
2. **Part 2 (only if Part 1 isn't good enough)**: fine-tune on your own labeled images.
   Captions are auto-generated from a short per-cluster template -- not hand-written --
   so this doesn't add a second giant prompt list to maintain alongside `CLUSTER_PROMPTS`.

Runtime: GPU (training needs it; captioning alone would work on CPU too, just slower).

In [ ]:
# Lesson from the CLIP notebook: don't touch torch/torchvision/pillow -- Colab's
# preinstalled versions are already matched. Only add what's actually missing.
!pip install -q "transformers==4.46.3"

## Part 1 -- stock BLIP-base captioning (no training)

In [ ]:
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "Salesforce/blip-image-captioning-base"

processor = BlipProcessor.from_pretrained(MODEL_ID)
model = BlipForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE)
print("Loaded on", DEVICE)

In [ ]:
!pip install -q kaggle
from google.colab import files
files.upload()  # kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d tarundalal/indian-temples-dataset -p data --unzip

In [ ]:
import os, random
from PIL import Image

def find_dataset_root(base="data"):
    for dirpath, dirnames, _ in os.walk(base):
        if len(dirnames) > 1:
            return dirpath
    return base

root = find_dataset_root()
all_images = [os.path.join(dp, f) for dp, _, fs in os.walk(root) for f in fs
              if f.lower().endswith((".jpg", ".jpeg", ".png"))]
random.seed(42)
sample_paths = random.sample(all_images, min(10, len(all_images)))
print(f"{len(all_images)} images found, sampling {len(sample_paths)}")

In [ ]:
@torch.no_grad()
def caption(image, prompt=None, max_new_tokens=30):
    if prompt:
        inputs = processor(image, prompt, return_tensors="pt").to(DEVICE)
    else:
        inputs = processor(image, return_tensors="pt").to(DEVICE)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return processor.decode(out[0], skip_special_tokens=True)

for path in sample_paths[:5]:
    image = Image.open(path).convert("RGB")
    unconditioned = caption(image)
    primed = caption(image, prompt="a photograph of a Hindu")
    print(f"\n{os.path.basename(path)}")
    print(f"  unconditioned: {unconditioned}")
    print(f"  primed:        {primed}")

**Check before going further**: do these captions name anything useful (temple, deity, ritual
words) that `_keyword_score()` could match on? If unconditioned/primed captions are already
saying things like "temple", "statue", "shrine" -- stock BLIP-base might be enough paired with
the existing keyword dictionary, no training needed. If captions are generic ("a building",
"a person") with no devotional vocabulary at all, that's the signal to fine-tune.

## Part 2 -- fine-tune on your own labeled images
Only run this if Part 1's captions came back too generic.

**Folder structure expected** (same `ImageFolder`-style layout as the CLIP notebooks) --
one folder per cluster, using the same cluster names as `CLUSTER_PROMPTS` in
`vision/clip_model.py` (e.g. `shiva_iconography/`, `krishna_iconography/`,
`yoga_and_meditation/`, ...). Upload a zip and point `DATA_ROOT` at it, or mount Drive.

In [ ]:
# EDIT ME
DATA_ROOT = "/content/devotional_images"  # one subfolder per cluster, your own images

# One short caption template per cluster -- deliberately ONE sentence each, not the full
# 5-10 prompt variants from CLUSTER_PROMPTS. Fine-tuning needs a caption target, not a
# discriminative prompt bank -- keeping this list separate and minimal avoids turning into
# a second thing to maintain alongside the CLIP prompts.
CAPTION_TEMPLATES = {
    "shiva_iconography": "a photograph of a sacred Shivalinga idol in a Hindu temple",
    "krishna_iconography": "a photograph of Lord Krishna playing a flute",
    "vishnu_iconography": "a photograph of the four-armed Hindu deity Lord Vishnu",
    "venkateswara_iconography": "a photograph of the Tirupati Balaji idol with a gold crown",
    "rama_iconography": "a photograph of Lord Rama holding a bow",
    "hanuman_iconography": "a photograph of Lord Hanuman in a devotional posture",
    "ganesha_iconography": "a photograph of the elephant-headed Hindu deity Lord Ganesha",
    "devi_iconography": "a photograph of a Hindu goddess idol wearing a gold crown",
    "puja_rituals_and_temples": "a photograph of a Hindu temple with devotees performing a puja ritual",
    "festivals_and_celebrations": "a photograph of people celebrating a Hindu festival",
    "yoga_and_meditation": "a photograph of a person performing a yoga pose or meditating",
    "prasad_and_sacred_food": "a photograph of temple prasadam food offering",
}

import glob

pairs = []  # (image_path, caption)
for cluster_name, cap in CAPTION_TEMPLATES.items():
    folder = os.path.join(DATA_ROOT, cluster_name)
    if not os.path.isdir(folder):
        continue
    for path in glob.glob(os.path.join(folder, "*")):
        pairs.append((path, cap))

print(f"{len(pairs)} (image, caption) pairs found across {len(CAPTION_TEMPLATES)} clusters")
if not pairs:
    print(f"Nothing found under {DATA_ROOT} -- upload your images there before continuing.")

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CaptionDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        path, cap = self.pairs[idx]
        image = Image.open(path).convert("RGB")
        return image, cap

def collate(batch):
    images, caps = zip(*batch)
    enc = processor(images=list(images), text=list(caps), return_tensors="pt", padding=True)
    enc["labels"] = enc["input_ids"].clone()
    return {k: v.to(DEVICE) for k, v in enc.items()}

train_ds = CaptionDataset(pairs)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate)

In [ ]:
EPOCHS = 3  # a few hundred images at this size doesn't need more; watch the loss curve

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(**batch)
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    print(f"epoch {epoch+1}/{EPOCHS}  mean_loss={total_loss/len(train_loader):.4f}")

model.eval()

## Compare: stock vs fine-tuned captions on held-out images

In [ ]:
stock_model = BlipForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE).eval()

@torch.no_grad()
def caption_with(m, image, max_new_tokens=30):
    inputs = processor(image, return_tensors="pt").to(DEVICE)
    out = m.generate(**inputs, max_new_tokens=max_new_tokens)
    return processor.decode(out[0], skip_special_tokens=True)

# EDIT ME: point at a few held-out devotional images not used in training
HELD_OUT_IMAGES = []  # e.g. glob.glob("/content/devotional_images_test/*/*")

for path in HELD_OUT_IMAGES[:10]:
    image = Image.open(path).convert("RGB")
    print(f"\n{path}")
    print(f"  stock:      {caption_with(stock_model, image)}")
    print(f"  fine-tuned: {caption_with(model, image)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
model.save_pretrained('/content/drive/MyDrive/blip_devotional_finetuned')
processor.save_pretrained('/content/drive/MyDrive/blip_devotional_finetuned')
print("Saved to /content/drive/MyDrive/blip_devotional_finetuned")